In [1]:
# Cell 1 - Import libraries
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [2]:
# Cell 2 - Create movie dataset
# Real movie data for recommendation system

movies_data = {
    'title': [
        'Stree 3', 'Pushpa 3', 'King', 'Mirzapur 4', 'Panchayat 4',
        'Fukrey 4', 'Bhool Bhulaiyaa 4', 'War 3', 'Tiger 4', 'Don 3',
        'Avengers 5', 'Spider-Man 4', 'Fast X Part 2', 'John Wick 5', 'Top Gun 3',
        'Inception 2', 'Interstellar 2', 'Batman Beyond', 'Superman Legacy', 'Wonder Woman 3',
        'Sacred Games 3', 'Delhi Crime 3', 'Scam 3', 'Kota Factory 4', 'TVF Pitchers 3',
        'Lockup 2', 'Bigg Boss 19', 'KBC 17', 'Shark Tank India 4', 'MasterChef India 8',
        'Ramayana', 'Sikandar', 'Jigra 2', 'Rocky Aur Rani 2', 'Ae Dil Hai Mushkil 2',
        'Golmaal 6', 'Housefull 6', 'Welcome 4', 'Munnabhai 3', 'Dhamaal 5',
        'RRR 2', 'KGF 3', 'Baahubali 4', 'Vikram 2', 'Varisu 2',
        'Ponniyin Selvan 3', 'Master 2', 'Beast 2', 'Jailer 2', 'Leo 2'
    ],
    'genre': [
        'Horror Comedy', 'Action', 'Action', 'Crime Thriller', 'Comedy Drama',
        'Comedy', 'Horror Comedy', 'Action', 'Action', 'Action',
        'Superhero', 'Superhero', 'Action', 'Action', 'Action',
        'Sci-Fi', 'Sci-Fi', 'Superhero', 'Superhero', 'Superhero',
        'Crime Thriller', 'Crime Drama', 'Drama', 'Drama', 'Drama',
        'Reality', 'Reality', 'Game Show', 'Reality', 'Reality',
        'Drama', 'Action', 'Drama', 'Romance', 'Romance',
        'Comedy', 'Comedy', 'Comedy', 'Comedy', 'Comedy',
        'Action', 'Action', 'Action', 'Action', 'Drama',
        'Drama', 'Action', 'Action', 'Action', 'Action'
    ],
    'platform': [
        'Theatre', 'Theatre', 'Theatre', 'Amazon Prime', 'Amazon Prime',
        'Theatre', 'Theatre', 'Theatre', 'Theatre', 'Theatre',
        'Theatre', 'Theatre', 'Theatre', 'Theatre', 'Theatre',
        'Theatre', 'Theatre', 'Theatre', 'Theatre', 'Theatre',
        'Netflix', 'Netflix', 'Netflix', 'Netflix', 'Netflix',
        'Netflix', 'Hotstar', 'Hotstar', 'Amazon Prime', 'Hotstar',
        'Theatre', 'Theatre', 'Theatre', 'Theatre', 'Theatre',
        'Theatre', 'Theatre', 'Theatre', 'Theatre', 'Theatre',
        'Theatre', 'Theatre', 'Theatre', 'Theatre', 'Theatre',
        'Theatre', 'Theatre', 'Theatre', 'Theatre', 'Theatre'
    ],
    'rating': [
        8.2, 8.9, 9.1, 8.8, 9.0,
        7.8, 7.5, 8.0, 7.8, 8.2,
        8.5, 8.3, 7.5, 8.0, 8.5,
        9.3, 9.0, 8.2, 7.8, 7.5,
        8.5, 8.4, 8.7, 9.0, 8.8,
        7.5, 6.8, 7.2, 8.0, 7.5,
        8.8, 8.5, 8.0, 7.9, 7.8,
        7.5, 7.0, 7.2, 8.0, 7.5,
        8.8, 8.9, 8.5, 8.7, 8.0,
        8.5, 8.6, 7.8, 8.9, 8.5
    ],
    'mood': [
        'Funny', 'Action', 'Action', 'Thriller', 'Emotional',
        'Funny', 'Horror', 'Action', 'Action', 'Action',
        'Action', 'Action', 'Action', 'Action', 'Action',
        'Drama', 'Drama', 'Action', 'Action', 'Action',
        'Thriller', 'Thriller', 'Drama', 'Emotional', 'Drama',
        'Funny', 'Funny', 'Funny', 'Drama', 'Funny',
        'Emotional', 'Action', 'Emotional', 'Romance', 'Romance',
        'Funny', 'Funny', 'Funny', 'Funny', 'Funny',
        'Action', 'Action', 'Action', 'Action', 'Emotional',
        'Emotional', 'Action', 'Action', 'Action', 'Action'
    ],
    'language': [
        'Hindi', 'Telugu', 'Hindi', 'Hindi', 'Hindi',
        'Hindi', 'Hindi', 'Hindi', 'Hindi', 'Hindi',
        'English', 'English', 'English', 'English', 'English',
        'English', 'English', 'English', 'English', 'English',
        'Hindi', 'Hindi', 'Hindi', 'Hindi', 'Hindi',
        'Hindi', 'Hindi', 'Hindi', 'Hindi', 'Hindi',
        'Hindi', 'Hindi', 'Hindi', 'Hindi', 'Hindi',
        'Hindi', 'Hindi', 'Hindi', 'Hindi', 'Hindi',
        'Telugu', 'Kannada', 'Telugu', 'Tamil', 'Tamil',
        'Tamil', 'Tamil', 'Tamil', 'Tamil', 'Tamil'
    ]
}

df = pd.DataFrame(movies_data)
print(f"Total movies: {len(df)}")
print(f"\nMood distribution:")
print(df['mood'].value_counts())
print("\n✅ Movie dataset created!")

Total movies: 50

Mood distribution:
mood
Action       22
Funny        11
Emotional     6
Drama         5
Thriller      3
Romance       2
Horror        1
Name: count, dtype: int64

✅ Movie dataset created!


In [3]:
# Cell 3 - Build recommendation system using TF-IDF + Cosine Similarity
# This is content based filtering - Ch 25-27 of your syllabus!

# Combine features into one text column
df['features'] = df['genre'] + ' ' + df['platform'] + ' ' + df['mood'] + ' ' + df['language']

# Create TF-IDF matrix
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['features'])

# Calculate cosine similarity between all movies
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Cosine similarity matrix shape: {cosine_sim.shape}")

# Function to get recommendations
def get_recommendations(movie_title, num_recommendations=5):
    # Find movie index
    if movie_title not in df['title'].values:
        return []
    
    idx = df[df['title'] == movie_title].index[0]
    
    # Get similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort by similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get top recommendations (exclude the movie itself)
    sim_scores = sim_scores[1:num_recommendations+1]
    
    # Get movie indices
    movie_indices = [i[0] for i in sim_scores]
    
    # Return recommended movies
    recommendations = []
    for i in movie_indices:
        recommendations.append({
            'title': df['title'][i],
            'genre': df['genre'][i],
            'platform': df['platform'][i],
            'rating': df['rating'][i],
            'mood': df['mood'][i]
        })
    
    return recommendations

# Test recommendations
print("\n=== TESTING RECOMMENDATIONS ===")
test_movies = ['Stree 3', 'Pushpa 3', 'Panchayat 4']

for movie in test_movies:
    print(f"\nBecause you liked '{movie}', watch:")
    recs = get_recommendations(movie, 3)
    for i, rec in enumerate(recs, 1):
        print(f"  {i}. {rec['title']} ({rec['genre']}) - ⭐{rec['rating']}")

print("\n✅ Recommendation system working!")

TF-IDF matrix shape: (50, 25)
Cosine similarity matrix shape: (50, 50)

=== TESTING RECOMMENDATIONS ===

Because you liked 'Stree 3', watch:
  1. Bhool Bhulaiyaa 4 (Horror Comedy) - ⭐7.5
  2. Fukrey 4 (Comedy) - ⭐7.8
  3. Golmaal 6 (Comedy) - ⭐7.5

Because you liked 'Pushpa 3', watch:
  1. RRR 2 (Action) - ⭐8.8
  2. Baahubali 4 (Action) - ⭐8.5
  3. King (Action) - ⭐9.1

Because you liked 'Panchayat 4', watch:
  1. Shark Tank India 4 (Reality) - ⭐8.0
  2. Ramayana (Drama) - ⭐8.8
  3. Jigra 2 (Drama) - ⭐8.0

✅ Recommendation system working!


In [4]:
# Cell 4 - K-Means Clustering
# Groups similar movies into clusters - Ch 22-24 of your syllabus!

from sklearn.preprocessing import LabelEncoder

# Encode categorical columns
le = LabelEncoder()
df['genre_enc'] = le.fit_transform(df['genre'])
df['platform_enc'] = le.fit_transform(df['platform'])
df['mood_enc'] = le.fit_transform(df['mood'])
df['language_enc'] = le.fit_transform(df['language'])

# Features for clustering
X_cluster = df[['genre_enc', 'platform_enc', 'mood_enc', 'language_enc', 'rating']]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Apply K-Means clustering
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Evaluate clustering
sil_score = silhouette_score(X_scaled, df['cluster'])
print(f"Silhouette Score: {sil_score:.3f}")
print("(Score closer to 1 = better clustering)")

# Show clusters
print("\n=== MOVIE CLUSTERS ===")
for cluster in range(5):
    cluster_movies = df[df['cluster'] == cluster]['title'].tolist()
    print(f"\nCluster {cluster}: {cluster_movies[:5]}")

print("\n✅ K-Means clustering complete!")

Silhouette Score: 0.363
(Score closer to 1 = better clustering)

=== MOVIE CLUSTERS ===

Cluster 0: ['King', 'War 3', 'Tiger 4', 'Don 3', 'Fast X Part 2']

Cluster 1: ['Stree 3', 'Fukrey 4', 'Bhool Bhulaiyaa 4', 'Sacred Games 3', 'Delhi Crime 3']

Cluster 2: ['Pushpa 3', 'RRR 2', 'KGF 3', 'Baahubali 4', 'Vikram 2']

Cluster 3: ['Avengers 5', 'Spider-Man 4', 'Inception 2', 'Interstellar 2', 'Batman Beyond']

Cluster 4: ['Mirzapur 4', 'Panchayat 4', 'Bigg Boss 19', 'KBC 17', 'Shark Tank India 4']

✅ K-Means clustering complete!


In [5]:
# Cell 5 - Get recommendations by mood
# This is what React will call!

def recommend_by_mood(mood, num=5):
    mood_movies = df[df['mood'] == mood].copy()
    mood_movies = mood_movies.sort_values('rating', ascending=False)
    
    recommendations = []
    for _, row in mood_movies.head(num).iterrows():
        recommendations.append({
            'title': row['title'],
            'genre': row['genre'],
            'platform': row['platform'],
            'rating': row['rating'],
            'mood': row['mood'],
            'language': row['language']
        })
    return recommendations

# Test mood recommendations
print("=== MOOD BASED RECOMMENDATIONS ===")
moods = ['Action', 'Funny', 'Emotional', 'Thriller', 'Romance']

for mood in moods:
    print(f"\n{mood} mood picks:")
    recs = recommend_by_mood(mood, 3)
    for rec in recs:
        print(f"  → {rec['title']} ({rec['platform']}) ⭐{rec['rating']}")

print("\n✅ Mood recommendations working!")

=== MOOD BASED RECOMMENDATIONS ===

Action mood picks:
  → King (Theatre) ⭐9.1
  → Pushpa 3 (Theatre) ⭐8.9
  → KGF 3 (Theatre) ⭐8.9

Funny mood picks:
  → Stree 3 (Theatre) ⭐8.2
  → Munnabhai 3 (Theatre) ⭐8.0
  → Fukrey 4 (Theatre) ⭐7.8

Emotional mood picks:
  → Panchayat 4 (Amazon Prime) ⭐9.0
  → Kota Factory 4 (Netflix) ⭐9.0
  → Ramayana (Theatre) ⭐8.8

Thriller mood picks:
  → Mirzapur 4 (Amazon Prime) ⭐8.8
  → Sacred Games 3 (Netflix) ⭐8.5
  → Delhi Crime 3 (Netflix) ⭐8.4

Romance mood picks:
  → Rocky Aur Rani 2 (Theatre) ⭐7.9
  → Ae Dil Hai Mushkil 2 (Theatre) ⭐7.8

✅ Mood recommendations working!


In [6]:
# Cell 6 - Save all models

models_dir = os.path.join(os.getcwd(), 'models')
os.makedirs(models_dir, exist_ok=True)

# Save cosine similarity matrix
sim_path = os.path.join(models_dir, 'cosine_sim.pkl')
with open(sim_path, 'wb') as f:
    pickle.dump(cosine_sim, f)
print("✅ Cosine similarity saved!")

# Save movie dataframe
df_path = os.path.join(models_dir, 'movies_df.pkl')
with open(df_path, 'wb') as f:
    pickle.dump(df, f)
print("✅ Movies dataframe saved!")

# Save KMeans model
kmeans_path = os.path.join(models_dir, 'kmeans_model.pkl')
with open(kmeans_path, 'wb') as f:
    pickle.dump(kmeans, f)
print("✅ KMeans model saved!")

# Save scaler
scaler_path = os.path.join(models_dir, 'scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print("✅ Scaler saved!")

# Save TF-IDF vectorizer
tfidf_path = os.path.join(models_dir, 'recommender_tfidf.pkl')
with open(tfidf_path, 'wb') as f:
    pickle.dump(tfidf, f)
print("✅ TF-IDF vectorizer saved!")

print("\n=== ALL MODELS SAVED ===")
print("\nFiles in models folder:")
for file in os.listdir(models_dir):
    print(f"  ✅ {file}")

print("\n🎉 Movie Recommender is complete!")

✅ Cosine similarity saved!
✅ Movies dataframe saved!
✅ KMeans model saved!
✅ Scaler saved!
✅ TF-IDF vectorizer saved!

=== ALL MODELS SAVED ===

Files in models folder:
  ✅ cosine_sim.pkl
  ✅ hitflop_model.pkl
  ✅ kmeans_model.pkl
  ✅ le_genre.pkl
  ✅ le_platform.pkl
  ✅ le_result.pkl
  ✅ movies_df.pkl
  ✅ recommender_tfidf.pkl
  ✅ scaler.pkl
  ✅ sentiment_model.pkl
  ✅ vectorizer.pkl

🎉 Movie Recommender is complete!
